# Checking to make sure Connecticut Planning Regions not present in original FIPS source

In [ ]:
import pandas as pd

################ BFS GEOGRAPHIC INDEX ###################
# Extracting geo_idx key from Census Business Formation Statistics (BFS) Complete Time Series Dataset
bfs = pd.read_csv('../data_raw/BFS-mf/BFS-mf.csv', skiprows=43, nrows=57)
bfs = bfs.rename(columns={'geo_code': 'STATE'})
print(bfs.head())



################# State FIPS Codes ###################
# Reading in State FIPS Codes from Census.gov
state_fips = pd.read_csv(
    "https://www2.census.gov/geo/docs/reference/codes2020/national_state2020.txt",
    sep="|", 
        dtype={
        "STATEFP": str
    })
print(state_fips.head())

# Merging with bfs geo_idx key by State Abreviation ("STATE")
state_id = pd.merge(state_fips, bfs, on="STATE", how='outer', indicator = True)

# Dropping extra columns and updating State FIPS variable name
state_id = state_id.drop(columns = ['geo_desc', 'STATENS'])
state_id = state_id.rename(columns = {'STATEFP': 'state_fips'})

# Saving resulting file to merge with intermediate files
print(state_id.head())
state_id.to_csv('../data_intermediate/state_geo_id.csv', index=False)



################# County FIPS Codes ###################
# Reading in County FIPS Codes from Census.gov
county_fips = pd.read_csv(
    "https://www2.census.gov/geo/docs/reference/codes2020/national_county2020.txt",
    sep="|",
    dtype={
        "STATEFP": str,
        "COUNTYFP": str
    })

# Reading in 

# Updating variable names and dropping unnecessary variables
county_fips = county_fips.rename(columns = {'STATEFP': 'state_fips', 'COUNTYFP': 'county_fips', 'COUNTYNAME': 'COUNTY_NAME'})
county_fips = county_fips.drop(columns = ['CLASSFP', 'FUNCSTAT', 'COUNTYNS'])
print(county_fips.head())

# Creating county-level geographic index file with County FIPS, State FIPS, and BFS State 'geo_idx'
county_id = pd.merge(county_fips, state_id, on='state_fips', how='outer', indicator = True)

# Dropping unnecessary columns and standardizing variable names
county_id = county_id.drop(columns = ['STATE_y'])
county_id = county_id.rename(columns = {'STATE_x': 'STATE'})
print(county_id.head())



################# Creating Unique County 5-digit FIPS Codes ###################
# Adding combined 5 digit state+county fips for unique a county identifier
county_id['full_fips'] = county_id['state_fips'] + county_id['county_fips']
print(county_id.head())

# Saving resulting file to merge with intermediate data
county_id.to_csv('../data_intermediate/county_geo_id.csv', index=False)

   geo_idx STATE    geo_desc
0        1    US  U.S. Total
1        2    NO   Northeast
2        3    MW     Midwest
3        4    SO       South
4        5    WE        West
  STATE STATEFP  STATENS  STATE_NAME
0    AL      01  1779775     Alabama
1    AK      02  1785533      Alaska
2    AZ      04  1779777     Arizona
3    AR      05    68085    Arkansas
4    CA      06  1779778  California
  STATE state_fips      STATE_NAME  geo_idx     _merge
0    AK         02          Alaska      6.0       both
1    AL         01         Alabama      7.0       both
2    AR         05        Arkansas      8.0       both
3    AS         60  American Samoa      NaN  left_only
4    AZ         04         Arizona      9.0       both
  STATE state_fips county_fips     COUNTY_NAME
0    AL         01         001  Autauga County
1    AL         01         003  Baldwin County
2    AL         01         005  Barbour County
3    AL         01         007     Bibb County
4    AL         01         009   Blount

ValueError: Cannot use name of an existing column for indicator column

## Checking Different Numbers of Obs Post-Merges

- Changed merges to include how='outer' option in code above
- None of these are geographic subdivisions I need in this analysis, so droppping them will be the best 
- Unfortunately no FIPS codes for Connecticut Planning Regions

-



In [ ]:
print(county_id[county_id['_merge'] != 'both'])